#### **Table 1.** Processing 1500 $f(x)=x^2$ operations in the Templet and Everest systems with 500 slots

| mode     |    1<sup>st</sup> try time, $s$  | 2<sup>nd</sup> try time, $s$ | 3<sup>rd</sup> try time, $s$  | mean time, $s$  | throughput, $s^{-1}$ | turnaround time, $s$|
|----------|-----------------:|-------------:|--------------:|-------------:|----------------:|-----------------:|
|Everest(raw) <sup>(1)</sup> | 346.558 | 348.601 |347.816 |347.658 | 4.314 |115.886 |
|Everest(parcels) <sup>(2)</sup> | 92.278| 97.167 | 91.421 | 93.622 | 16.021 | 31.207 | 
|Templet(sync) <sup>(3)</sup> | 12.783 | 17.124 | 11.284 | 13.731 | 109.241 |4.577 |
|Templet(lasy) <sup>(4)</sup> | 0.322 | 0.658 | 1.119 | 0.700 | 2141.832 | 0.233 |
-------------------
<sup>**(1)**</sup> Everest(raw): еach $f(x)=x^2$ operation is performed in a separate task, 1500 tasks in total.
<sup>**(2)**</sup> Everest(parcels): in one task three $f(x)=x^2$ operations are performed, 500 tasks in total.
<sup>**(3)**</sup> Templet(sync): data is flushed to disk every time an event log write is performed.
<sup>**(4)**</sup> Templet(lasy): writing data to disk is performed in a delayed mode.

#### **Table 2.** Processing 15000 $f(x)=x^2$ operations in the Templet and Everest systems with 500 slots

| mode     |    1<sup>st</sup> try time, $s$  | 2<sup>nd</sup> try time, $s$ | 3<sup>rd</sup> try time, $s$  | mean time, $s$  | throughput, $s^{-1}$ | turnaround time, $s$|
|----------|-----------------:|-------------:|--------------:|-------------:|----------------:|-----------------:|
|Everest(parcel) <sup>(1)</sup>|	93.253|	98.394|	94.316|	95.321|	157.362|	3.177|
|Templet(sync) <sup>(2)</sup> |	186.866|	236.094|	260.207|	227.722|	65.869|	7.590|
|Templet(lasy) <sup>(3)</sup> |	32.193|	176.583|	74.403|	94.393|	158.909|	3.146|
|Templet(recover) <sup>(4)</sup> |	1.327|	1.776|	1.050|	1.384|	10831.732|	0.046|
------------------------
<sup>**(1)**</sup> Everest(parcel): in one task 30 $f(x)=x^2$ operations are performed, 500 tasks in total.
<sup>**(2)**</sup> Templet(sync): data is flushed to disk every time an event log write is performed.
<sup>**(3)**</sup> Templet(lasy): writing data to disk is performed in a delayed mode.
<sup>**(4)**</sup> Templet(recover): time to read 15,000 previously completed tasks and restore the final state.

#### **Figure 1.** Interface and state of the *bag-of-tasks* at the basic stage of program design

In [ ]:
class base_bag_of_tasks{
public:
    void resize(unsigned size){ <...> }
    void add(unsigned id,int n){ <...> }
    bool ready_to_get(){ <...> }
    bool get(unsigned& id, int& n){ <...> }
    void put(unsigned id,int nxn){ <...> }
public:
    std::vector<int> N;
    std::vector<int> NxN;
private:
    std::set<unsigned> unprocessed;
    bool _ready_to_get;
    unsigned get_rand_unprocessed(){ <...> }
};

#### **Figure 2.** The basic program for applying $f(x)=x^2$ to array of numbers

In [ ]:
    base_bag_of_tasks tbag;
    
    tbag.resize(SIZE);
    for(int i=0;i<SIZE;i++) tbag.add(i,i);
    
    while(!tbag.ready_to_get())/*wait*/;
    
    unsigned id; int N, NxN;
    while(tbag.get(id,N)){
        NxN = N*N; tbag.put(id,NxN);
    }
    for(int i=0;i<SIZE;i++)
        std::cout << tbag.N[i] <<"^2 = " 
                  << tbag.NxN[i] << std::endl;

#### **Figure 3.** Modifying the base methods of the *bag-of-tasks* class to achieve synchronized state behavior

In [ ]:
void base_bag_of_tasks::add(unsigned id,int n){
    N[id]=n; unprocessed.insert(id);
    if(unprocessed.size()==N.size())_ready_to_get=true;
} //--- modified to -->
void bag_of_tasks::add(unsigned id,int n){
    update(_add, [&](std::ostream&out) {
        out << id << " " << n;
    },
    [this](std::istream&in) {
        unsigned id; int n; in >> id >> n;
        N[id]=n; unprocessed.insert(id);
        if(unprocessed.size()==N.size())_ready_to_get=true;
    });     
}

#### **Figure 4.** Modification of the basic sequential program to obtain the final distributed program

In [ ]:
    bag_of_tasks tbag(wal); 
    if(pid==0 && !tbag.ready_to_get()){// in master 'process'
        tbag.resize(SIZE);
        for(int i=0;i<SIZE;i++) tbag.add(i,i);
    }
    while(!tbag.ready_to_get())/*wait*/;
    
    unsigned id; int N, NxN; 
    while(tbag.get(id,N)){
        NxN = N*N; tbag.put(id,NxN);
    }
    if(pid==0){// in master 'process'
        for(int i=0;i<SIZE;i++) std::cout << tbag.N[i]
            <<"^2 = " << tbag.NxN[i] << std::endl;
    }

#### **Figure 5.** We use the dependency injection pattern to change the implementation method without changing the algorithm

In [ ]:
class mapper{
public:
    class engine{
        void on_init(unsigned size){_mapper->on_init(size);}  <...>
    protected:
        mapper* _mapper;
    };
public:
    mapper(mapper::engine& eng):_engine(eng){eng._mapper=this;}
    void init(unsigned size){_engine.init(size);}   <...>
protected:
    virtual void on_init(unsigned size) = 0;        <...>
private:
    mapper::engine& _engine;
};

#### **Figure 6.** A basic mechanism for managing element-wise iteration and applying abstract operations

In [ ]:
class basic_engine: public mapper::engine{
    void init(unsigned size)override{_size=size;on_init(size);}
    void map()override{
        for(int id=0;id<_size;id++)
            {io_test(id,false); on_map(id); io_test(id,true);}
    }
    void io_test(unsigned id,bool mapped){
        std::stringstream sstr;
        on_save(id,sstr,mapped);on_load(id,sstr,mapped);
    }
    unsigned _size;
};
class sync_state_engine: public mapper::engine{ <...> };
class everest_engine: public mapper::engine{ <...> };

#### **Figure 7.** Element-wise application of a function $f(x)=x^2$ using the universal *mapper* algorithm

In [ ]:
class throughput_test_mapper: public mapper{
public:
    throughput_test_mapper(mapper::engine& eng): mapper(eng){}
    std::vector<int> N; std::vector<int> NxN;
private:
    void on_init(unsigned size) override{N.resize(size);NxN.resize(size);}
    void on_map(unsigned id) override{ NxN[id] = N[id] * N[id]; }
    void on_save(unsigned id, std::ostream& out, bool mapped) override{
        if(mapped) out << NxN[id]; else out << N[id];  
    }
	void on_load(unsigned id, std::istream& in, bool mapped) override{
        if(mapped) in >> NxN[id]; else in >> N[id];
    }
};

#### **Figure 8.** A test program for element-by-element application of the function $f(x)=x^2$ using various implementations

In [ ]:
    basic_engine eng;
    //sync_state_engine eng;
    //everest_engine eng;  
    throughput_test_mapper a_mapper(eng);   
    if(master){
        a_mapper.N.resize(SIZE);
        for(int i=0;i<SIZE;i++) a_mapper.N[i] = i;
        a_mapper.init(SIZE);
    } 
    a_mapper.map(); 
    if(master){
        for(int i=0;i<SIZE;i++)
            std::cout << a_mapper.N[i] 
                <<"^2 = " << a_mapper.NxN[i] << std::endl;
    }